# SnowHealth Demand Forecasting — Feature Engineering

Transforms raw transactions into ML-ready features and registers them in the **Snowflake Feature Store**.

| Step | What happens |
|---|---|
| Engineer features | Calendar, cyclical, lag/rolling, and interaction features via `source/feature_engineering.py` |
| Save to Snowflake | Write features table with change tracking enabled |
| Feature Store | Register entity + managed Feature View (Dynamic Table, 10-min refresh) |
| Online serving | Enable low-latency key-value lookups (1-min target lag) |
| Lineage | Verify feature-to-model lineage is tracked |

> Requires `01_setup.ipynb` to have been run first.

### Initialize Session
Sets up the Python path to import shared modules from `source/`, connects to the active Snowflake session, and switches context to the `DEV_SNOWHEALTH_ML_DB` database. Imports the `engineer_features` pipeline and `config` module used throughout this notebook.

In [ ]:
import sys, os
import pandas as pd
import numpy as np

# Resolve source/ (works from root or notebooks/ subdirectory)
for _p in [os.path.join(os.getcwd(), 'source'), os.path.join(os.getcwd(), '..', 'source')]:
    if os.path.isdir(_p): sys.path.insert(0, os.path.realpath(_p)); break

from snowflake.snowpark.context import get_active_session
import config as cfg
from feature_engineering import engineer_features, get_feature_columns

session = get_active_session()
session.sql(f"USE DATABASE {cfg.SNOWHEALTH_DATABASE}").collect()
session.sql(f"USE SCHEMA {cfg.SNOWHEALTH_RAW_SCHEMA}").collect()
print(f"Session ready: {cfg.SNOWHEALTH_DATABASE}")

### Load Raw Data and Engineer Features
Loads the 10,000-row transactions table created in `01_setup.ipynb` and runs the full feature engineering pipeline from `source/feature_engineering.py`. The pipeline applies four transform stages:

1. **Calendar features** — `MONTH`, `QUARTER`, `WEEK_OF_YEAR`, weekend/holiday/promotion flags
2. **Cyclical encoding** — `sin`/`cos` transforms of month and day-of-week so the model understands that December and January are adjacent
3. **Lag and rolling features** — 7/14/30-day lags and 7/30-day rolling averages per `SKU_ID × STORE_ID`, capturing recent demand trends
4. **Interaction features** — `PRICE_TIMES_PROMO` (price sensitivity during promotions), `INVENTORY_TURNOVER_RATIO`, and `SEASONAL_DEMAND_INDEX`

Categorical columns (`REGION`, `PRODUCT_CATEGORY`, `FULFILLMENT_CHANNEL`) are one-hot encoded into binary dummy columns.

In [ ]:
# Load raw data and apply the full feature engineering pipeline
# (calendar + cyclical + lag/rolling + interaction features — see source/feature_engineering.py)
df_raw = session.table(cfg.SNOWHEALTH_TRANSACTIONS_TABLE).to_pandas()
df_feat = engineer_features(df_raw, cfg.CATEGORICAL_FEATURES)

print(f"Raw: {df_raw.shape[1]} cols → Engineered: {df_feat.shape[1]} cols ({len(df_feat):,} rows)")
print(f"Target (UNITS_SOLD): mean={df_feat['UNITS_SOLD'].mean():.1f}, std={df_feat['UNITS_SOLD'].std():.1f}")

### Visualize Engineered Features
Spot-check distributions of key engineered features to verify they look reasonable before saving to Snowflake. We plot rolling averages, cyclical encodings, and interaction features — the features most likely to surface issues (e.g., all-zero lag columns, degenerate distributions).

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
matplotlib.use('Agg')

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
feats = ['ROLLING_7D_AVG_UNITS', 'ROLLING_30D_AVG_UNITS', 'SEASONAL_DEMAND_INDEX',
         'MONTH_SIN', 'INVENTORY_TURNOVER_RATIO', 'PRICE_TIMES_PROMO']
for i, feat in enumerate(feats):
    ax = axes[i//3, i%3]
    ax.hist(df_feat[feat].dropna(), bins=40, color='#29B5E8', edgecolor='white', alpha=0.8)
    ax.set_title(feat, fontsize=10)
plt.suptitle('Engineered Feature Distributions', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### Save Features to Snowflake
Persists the engineered feature DataFrame to `SNOWHEALTH_ML_DB.SNOWHEALTH_FEATURES.SNOWHEALTH_DEMAND_FEATURES`. Columns from the raw data that are not useful for modeling (`CREATED_AT`, `REVENUE`, raw boolean flags) are excluded.

An `EVENT_TIMESTAMP` column is added from `TRANSACTION_DATE` — this is required by the Feature Store for **point-in-time correct** feature retrieval during training (prevents data leakage by ensuring only features known at prediction time are used).

**Change tracking** is enabled on the table, which is a prerequisite for managed Feature Views (Dynamic Tables) to perform incremental refreshes instead of full recomputation.

In [ ]:
# Select columns to persist: IDs + target + all numeric features + EVENT_TIMESTAMP
# Note: engineer_features() one-hot encodes categoricals, so REGION/PRODUCT_CATEGORY/FULFILLMENT_CHANNEL
# are replaced by dummy columns (e.g., REG_SOUTHEAST, CAT_OTC_PAIN_RELIEF, CHAN_ONLINE_DELIVERY).

exclude_cols = {'CREATED_AT', 'REVENUE', 'IS_WEEKEND', 'IS_HOLIDAY', 'PROMOTION_FLAG'}
keep_cols = [c for c in df_feat.columns if c not in exclude_cols]

df_save = df_feat[keep_cols].copy().reset_index(drop=True)
df_save['TRANSACTION_DATE'] = pd.to_datetime(df_save['TRANSACTION_DATE'])
df_save['EVENT_TIMESTAMP'] = df_save['TRANSACTION_DATE']

# Widen narrow int/float types for Snowflake compatibility
for c in df_save.select_dtypes(include=['int8','int16','int32','uint8']).columns:
    df_save[c] = df_save[c].astype('int64')
for c in df_save.select_dtypes(include=['float32']).columns:
    df_save[c] = df_save[c].astype('float64')

# Write to Snowflake and enable change tracking (required for managed Feature Views / Dynamic Tables)
snow_df = session.create_dataframe(df_save)
snow_df.write.mode('overwrite').save_as_table(cfg.SNOWHEALTH_FEATURE_TABLE)

session.sql(f"ALTER TABLE {cfg.SNOWHEALTH_FEATURE_TABLE} SET CHANGE_TRACKING = TRUE").collect()

print(f"Saved {len(df_save):,} rows × {len(df_save.columns)} cols to {cfg.SNOWHEALTH_FEATURE_TABLE}")
print(f"Change tracking: enabled")

### Feature Store — Register Entity and Managed Feature View
The **Snowflake Feature Store** is a schema-level construct that provides centralized feature management, automated refresh, and lineage tracking.

**Key concepts:**
- **Entity** — the business object that features describe. Here it's the `SKU × Store` combination, with join keys `SKU_ID` and `STORE_ID`. Entities are immutable after registration.
- **Feature View** — a managed view over the features table, backed by a **Dynamic Table** that auto-refreshes every 10 minutes. Any changes to the upstream features table are automatically propagated.
- **Feature descriptions** — human-readable metadata attached to individual features for discoverability and documentation.

This cell initializes the Feature Store, registers the entity (idempotent), deletes any prior Feature View version (for re-runnability), then creates and registers the managed Feature View.

In [ ]:
# ----- Snowflake Feature Store ------------------------------------------------
# The Feature Store is a schema-level construct backed by Dynamic Tables.
# Entity = the business object (SKU × Store); Feature View = a managed DT that
# auto-refreshes from the features table we just created.

from snowflake.ml.feature_store import FeatureStore, FeatureView, Entity, CreationMode

# Connect to (or create) the feature store schema
fs = FeatureStore(
    session=session,
    database=cfg.SNOWHEALTH_DATABASE,
    name=cfg.SNOWHEALTH_FEATURES_SCHEMA,
    default_warehouse="COMPUTE_WH",
    creation_mode=CreationMode.CREATE_IF_NOT_EXIST,
)

# Register the entity (idempotent)
try:
    snowhealth_entity = fs.get_entity(cfg.SNOWHEALTH_ENTITY_NAME)
    print(f"Entity exists: {cfg.SNOWHEALTH_ENTITY_NAME}")
except:
    snowhealth_entity = Entity(
        name=cfg.SNOWHEALTH_ENTITY_NAME,
        join_keys=["SKU_ID", "STORE_ID"],
        desc="SKU-Store combination for demand forecasting"
    )
    fs.register_entity(snowhealth_entity)
    print(f"Entity registered: {cfg.SNOWHEALTH_ENTITY_NAME}")

# Delete any existing FV version so this cell is re-runnable
try:
    fs.delete_feature_view(cfg.SNOWHEALTH_FEATURE_VIEW_NAME, cfg.SNOWHEALTH_FEATURE_VIEW_VERSION)
    print(f"Deleted existing FV: {cfg.SNOWHEALTH_FEATURE_VIEW_NAME}/{cfg.SNOWHEALTH_FEATURE_VIEW_VERSION}")
except:
    pass

# Build and register the managed Feature View (creates a Dynamic Table with 10-min refresh)
feature_df = session.table(cfg.SNOWHEALTH_FEATURE_TABLE)

fv = FeatureView(
    name=cfg.SNOWHEALTH_FEATURE_VIEW_NAME,
    entities=[snowhealth_entity],
    feature_df=feature_df,
    timestamp_col="EVENT_TIMESTAMP",
    refresh_freq="10 minutes",
    desc="SnowHealth demand features — managed DT, 10-min refresh",
)

fv = fv.attach_feature_desc({
    "UNITS_SOLD": "Target: units sold per transaction",
    "ROLLING_7D_AVG_UNITS": "7-day rolling avg units sold per SKU-Store",
    "ROLLING_30D_AVG_UNITS": "30-day rolling avg units sold per SKU-Store",
    "SEASONAL_DEMAND_INDEX": "Monthly seasonal demand multiplier (0.4–1.6)",
    "INVENTORY_TURNOVER_RATIO": "Units sold / inventory on hand",
    "PRICE_TIMES_PROMO": "Unit price × promotion discount pct",
    "MONTH_SIN": "Cyclical month encoding (sine)",
    "MONTH_COS": "Cyclical month encoding (cosine)",
})

registered_fv = fs.register_feature_view(
    feature_view=fv,
    version=cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
    block=True,
)
print(f"\nFeature View registered: {registered_fv.name}/{registered_fv.version}")
print(f"Status: {registered_fv.status} | Type: Managed (Dynamic Table) | Refresh: 10 min")

### Enable Online Feature Serving (Hybrid Table)
Online feature serving provides **low-latency key-value lookups** for real-time inference. We use the **hybrid-table online store** (`OnlineStoreType.HYBRID_TABLE`), which keeps the latest feature values per `SKU_ID × STORE_ID` in a Snowflake hybrid table and syncs them from the offline feature view every minute (`target_lag="1m"`).

At inference time (notebook 05), the application looks up the latest features for an entity from the online store and sends them to the model service.

In [ ]:
from snowflake.ml.feature_store import OnlineConfig
from snowflake.ml.feature_store.feature_view import OnlineStoreType

# Hybrid-table online store: low-latency point lookups for the latest feature
# values per entity, kept in sync with the offline feature view on target_lag.
online_cfg = OnlineConfig(
    enable=True,
    target_lag="1m",
    store_type=OnlineStoreType.HYBRID_TABLE,
)
updated_fv = fs.update_feature_view(
    name=cfg.SNOWHEALTH_FEATURE_VIEW_NAME,
    version=cfg.SNOWHEALTH_FEATURE_VIEW_VERSION,
    online_config=online_cfg,
)
print(f"Hybrid-table online serving enabled (target lag: 1 min)")
print(f"Online store populates on the first sync — allow 1-2 minutes.")
print(f"Store type: {updated_fv.online_config}")

### Verify Feature Store Lineage
Snowflake ML automatically tracks **lineage** — the chain of dependencies from source tables through Feature Views, Datasets, and Models. This cell queries `SNOWFLAKE.CORE.GET_LINEAGE` to verify upstream edges from the Feature View back to the source features table.

The full lineage chain (Feature View → Dataset → Model) is established when:
1. Training data is generated via `fs.generate_dataset()` or `fs.retrieve_feature_values()` (notebook 03)
2. The trained model is registered in the Snowflake Model Registry (notebook 03)

Lineage metadata propagates asynchronously and may take up to 2 hours to appear. You can also view it visually in **Snowsight → Data → Lineage** tab.

In [ ]:
# ----- Lineage ----------------------------------------------------------------
# Snowflake ML tracks lineage automatically when:
#   1. Features are served via Feature Store (entity → feature view → feature table)
#   2. Training data is generated via fs.generate_dataset() or fs.retrieve_feature_values()
#   3. Models are registered in the Snowflake Model Registry
#
# The full chain (Feature View → Dataset → Model) is established in notebook 03.
# Here we verify the feature view's upstream lineage to the source features table.

lineage_fqn = f"{cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}.{cfg.SNOWHEALTH_FEATURE_VIEW_NAME}"

try:
    lineage_df = session.sql(f"""
        SELECT SOURCE_OBJECT_NAME, SOURCE_OBJECT_DOMAIN,
               TARGET_OBJECT_NAME, TARGET_OBJECT_DOMAIN
        FROM TABLE(
            SNOWFLAKE.CORE.GET_LINEAGE(
                '{lineage_fqn}', 'FEATURE_VIEW', 'UPSTREAM', 3
            )
        )
        LIMIT 20
    """).to_pandas()
    if len(lineage_df) > 0:
        print(f"Upstream lineage: {len(lineage_df)} edge(s)")
        print(lineage_df.to_string(index=False))
    else:
        print("No lineage edges yet — ACCOUNT_USAGE views may take up to 2 hours to populate.")
        print("Full lineage (FV → Dataset → Model) is established in notebook 03.")
except Exception as e:
    print(f"Lineage query note: {type(e).__name__}")
    print("Lineage metadata propagates asynchronously — run this cell again later or after notebook 03.")
    print("You can also view lineage visually in Snowsight: Data > Lineage tab.")

### Summary
Prints a recap of everything created in this notebook: the features table, Feature Store schema, registered entity and Feature View, online serving status, and total feature count. Confirms readiness for model training in `03_model_training.ipynb`.

In [ ]:
final_fv = fs.get_feature_view(cfg.SNOWHEALTH_FEATURE_VIEW_NAME, cfg.SNOWHEALTH_FEATURE_VIEW_VERSION)
print('=' * 60)
print('  FEATURE ENGINEERING COMPLETE')
print('=' * 60)
print(f"  Features table: {cfg.SNOWHEALTH_FEATURE_TABLE}")
print(f"  Feature Store:  {cfg.SNOWHEALTH_DATABASE}.{cfg.SNOWHEALTH_FEATURES_SCHEMA}")
print(f"  Entity:         {cfg.SNOWHEALTH_ENTITY_NAME} (SKU_ID, STORE_ID)")
print(f"  Feature View:   {final_fv.name}/{final_fv.version} — Managed DT, 10-min refresh")
print(f"  Online serving: {final_fv.online}")
print(f"  Features:       {len(final_fv.feature_names)}")
print('=' * 60)
print('  Next → 03_model_training.ipynb')